# 01. RAG란 무엇인가? — LLM에게 오픈북 시험을 보게 하기

이 노트북에서는 데이터베이스를 하나도 쓰지 않고, **순수 파이썬 몇 줄로 가장 단순한 RAG를 직접 만들어 봅니다.**
먼저 LLM이 논문 속 구체적인 수치를 물었을 때 어떻게 "그럴듯하게 틀리는지" 확인하고,
논문 조각을 찾아서 함께 건네주면 답이 어떻게 달라지는지 비교합니다.

**학습 목표**
- LLM이 "다음 단어 예측기"라는 것과, 그래서 생기는 **환각(Hallucination)** 을 이해합니다.
- **RAG(Retrieval-Augmented Generation, 검색 증강 생성)** 의 세 단계(검색 → 증강 → 생성)를 직접 구현합니다.
- 논문 PDF를 읽고 **조각(Chunk)** 으로 나누는 이유와 `chunk_size`, `chunk_overlap` 의 의미를 이해합니다.
- 키워드 검색의 한계(한국어 질문 vs 영어 논문)를 확인하고, 다음 노트북들에서 배울 DB가 왜 필요한지 알게 됩니다.

## 환경설정

In [ ]:
# [Colab 전용] 저장소를 내려받고 패키지·Ollama·모델을 설치합니다. (로컬 Jupyter에서는 자동으로 건너뜁니다)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/skku_rag_tutorials"):
        !git clone -q https://github.com/JDeun/skku_rag_tutorials.git /content/skku_rag_tutorials
    %cd /content/skku_rag_tutorials
    !bash scripts/colab_setup.sh

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # 실습과 무관한 경고 숨기기

from skku_rag import get_llm, load_papers, split_documents, LLM_MODEL, EMBED_MODEL
print("언어모델:", LLM_MODEL, "| 임베딩모델:", EMBED_MODEL)

llm = get_llm()  # 이 노트북 전체에서 쓸 언어모델

## 1. LLM이란? — "다음 단어 예측기"

**LLM(Large Language Model, 대규모 언어모델)** 이 하는 일은 놀랍도록 단순합니다.
> 지금까지 나온 글을 보고, **다음에 올 단어로 가장 그럴듯한 것**을 하나 고른다. 이것을 계속 반복한다.

```
입력:  "구리는 전기 전도도가 높아서 배선 재료로"
예측:   → "널리"(62%)  "많이"(21%)  "주로"(9%) ...
선택:  "구리는 전기 전도도가 높아서 배선 재료로 널리" → 다시 다음 단어 예측 → ...
```

**비유**: 수천만 편의 글을 읽으며 "빈칸 다음 단어 맞히기"만 연습한 학생입니다.
- 교과서에 수없이 나온 내용(구리의 비저항, 확산 방정식 등)은 **잘 기억합니다.**
- 하지만 특정 논문의 Table 1에 있는 숫자처럼 **드물게 본 내용은 흐릿하게만** 기억합니다.
- 문제는 이 학생이 "모르겠습니다"라고 말하는 법을 잘 배우지 못했다는 것입니다. 기억이 흐릿해도 **문장은 매끄럽게** 만들어 냅니다.

먼저 교과서 수준의 질문을 해 봅시다.

In [ ]:
# 교과서에 자주 나오는 일반 지식 질문
general_q = "구리(Cu)의 상온 벌크 비저항은 대략 얼마인가요? 한 문장으로 답해 주세요."
print(llm.invoke(general_q).content)

대부분의 모델은 일반적으로 알려진 값(약 1.7 μΩ·cm)에 가깝게 답합니다. 이런 지식은 학습 데이터에 수없이 나오기 때문입니다. 여러분의 결과도 이 값에 가까운지 확인해 보세요.

## 2. 논문 속 구체적 사실을 물어보면?

이번에는 실습 데이터인 Kim 등(2016, *Scripta Materialia*) 논문에만 나오는 **구체적인 수치**를 물어봅니다.
이 논문은 Cu에 Nb나 Fe를 아주 조금 넣어 스퍼터링한 나노결정 박막을 연구했는데,
"합금 원소를 넣으면 결정립이 작아진다"는 상식과 달리 **결정립이 오히려 커지는 농도 구간**을 발견했습니다.

In [ ]:
# 논문에만 나오는 구체적인 사실을 묻는 질문 (이 노트북 전체에서 계속 사용합니다)
QUESTION = (
    "Kim 등(2016, Scripta Materialia)의 스퍼터링 나노결정 Cu-Nb 박막 연구에서, "
    "결정립 크기가 순수 Cu보다 커지는(grain coarsening) Nb 농도 구간은 몇 at.%이고, "
    "평균 결정립 크기는 순수 Cu 대비 최대 몇 % 커졌나요? 3문장 이내로 답해 주세요."
)
answer_without_rag = llm.invoke(QUESTION).content  # 논문 없이 '기억'만으로 답하기
print(answer_without_rag)

논문 3쪽(115쪽)에 적힌 실제 정답은 다음과 같습니다.

> "Grain coarsening occurs in the range of **0 to 1.5 at.% Nb** and 0 to 2.5 at.% Fe ...
> whereas the mean grain size of the pure Cu film is **10.1 nm**, those of 1.14 at.% Nb ... increased to **13.6 nm (35% increase)**"

| 항목 | 논문의 정답 |
|---|---|
| 결정립이 커지는 Nb 농도 구간 | **0 ~ 1.5 at.%** |
| 평균 결정립 크기 변화 | 10.1 nm → 13.6 nm (1.14 at.% Nb), **35% 증가** |

위 LLM의 답과 비교해 보세요. 숫자가 다르거나, 논문에 없는 설명(메커니즘 등)을 **자신 있게** 덧붙이는 경우가 많습니다. (우연히 비슷하게 맞혔더라도 근거가 없다는 점은 같습니다.)
같은 셀을 다시 실행하면 **지어낸 숫자가 매번 달라지기도** 합니다. 근거 없이 기억에서 짜낸 답이라는 신호입니다.
이렇게 사실이 아닌 내용을 그럴듯하게 만들어 내는 현상을 **환각(Hallucination)** 이라고 부릅니다.

> 💡 **왜 이런 일이 생길까요?** LLM은 "정답"을 찾는 것이 아니라 "그럴듯한 다음 단어"를 고릅니다.
> "결정립 크기가 약 ___% 증가" 의 빈칸에는 어떤 숫자든 문법적으로는 자연스럽습니다.
> 특히 우리가 쓰는 4B(40억 파라미터) 소형 모델은 기억 용량이 작아 이런 일이 더 잦습니다. 큰 모델도 정도의 차이일 뿐 완전히 피하지는 못합니다.

## 3. 해결책: 오픈북 시험 — RAG

시험을 생각해 봅시다.
- **클로즈드북 시험**: 머릿속 기억만으로 답해야 합니다. 기억이 흐릿하면 그럴듯하게 지어냅니다. ← 지금까지의 LLM
- **오픈북 시험**: 문제와 관련된 교재 페이지를 **찾아서 펼쳐 놓고** 답합니다. ← RAG

RAG(Retrieval-Augmented Generation, 검색 증강 생성)는 이름 그대로 세 단계입니다.

```
  [질문] ──▶ ① 검색(Retrieval)     논문 더미에서 질문과 관련된 조각을 찾는다
                    │
                    ▼
            ② 증강(Augmented)     찾은 조각을 질문 앞에 붙여 '참고자료 + 질문' 프롬프트를 만든다
                    │
                    ▼
            ③ 생성(Generation)    LLM이 참고자료를 근거로 답을 만든다 ──▶ [답변]
```

실험실 비유로는, 새로 받은 시료의 XRD 패턴을 기억에 의존해 해석하는 대신
**데이터베이스에서 비슷한 피크를 가진 기준 패턴을 찾아(검색) 옆에 놓고(증강) 비교하며 판정(생성)** 하는 것과 같습니다.

이제 이 세 단계를 하나씩 직접 만들어 봅시다. 먼저 "교재", 즉 논문을 읽어야 합니다.

## 4. 논문 PDF 읽기

`load_papers()` 는 `paper_data/` 폴더의 PDF를 **페이지 단위**로 읽어 `Document` 목록으로 돌려줍니다.
`Document` 는 LangChain에서 쓰는 "글 한 덩어리" 상자로, 두 칸으로 되어 있습니다.
- `page_content`: 본문 텍스트
- `metadata`: 부가 정보(파일 경로, 페이지 번호 등) — 실험노트의 날짜·시료 번호 같은 라벨

In [ ]:
from collections import Counter
from pathlib import Path

docs = load_papers()  # 논문 2편 → 페이지 단위 Document 목록
print("전체 페이지 수:", len(docs))

# 논문(파일)별 페이지 수 세기 — 경로 전체 대신 파일 이름만 사용
for name, n in Counter(Path(d.metadata["source"]).name for d in docs).items():
    print(f"  {n:2d}쪽  {name}")

In [ ]:
page = docs[2]  # Kim 논문의 3번째 페이지 (파이썬은 0부터 셉니다)
print("파일:", Path(page.metadata["source"]).name, "| 페이지:", page.metadata["page"] + 1)
print("글자 수:", len(page.page_content))
print("-" * 60)
print(page.page_content[:600])  # 앞부분 600글자만 미리보기

PDF에서 뽑은 텍스트라 단어 중간에 줄바꿈(`slid-\ning`)이 있기도 하지만, 내용은 잘 읽혔습니다.
한 페이지에 수천 글자, 논문 두 편을 합치면 15만 글자가 넘습니다.

## 5. 조각내기(Chunking) — 왜 쪼갤까?

질문할 때마다 논문 전체를 LLM에게 통째로 주면 안 될까요? 세 가지 이유로 곤란합니다.
1. **입력 한도**: LLM이 한 번에 읽을 수 있는 양(컨텍스트 길이)에는 한계가 있습니다.
2. **속도·비용**: 입력이 길수록 답이 느려집니다. 내 컴퓨터의 소형 모델에서는 특히 체감됩니다.
3. **집중력**: 관련 없는 글이 많으면 정작 중요한 한 문장을 놓칩니다. (100쪽 실험노트를 통째로 건네며 "어제 소결 온도 몇 도였지?" 묻는 것과 같습니다.)

그래서 논문을 **적당한 크기의 조각(Chunk)** 으로 나눠 두고, 질문과 관련된 조각 몇 개만 골라서 건넵니다.

| 설정 | 의미 | 비유 |
|---|---|---|
| `chunk_size=1000` | 조각 하나의 최대 글자 수 | 실험노트를 **색인 카드** 크기로 자르기 |
| `chunk_overlap=100` | 이웃한 조각끼리 겹치는 글자 수 | 카드 경계에서 문장이 잘리지 않게 **앞 카드 끝 몇 줄을 다음 카드 머리에 다시 적기** |

In [ ]:
chunks = split_documents(docs, chunk_size=1000, chunk_overlap=100)  # 페이지 → 조각
print("페이지 수:", len(docs), "→ 조각 수:", len(chunks))

lengths = [len(c.page_content) for c in chunks]  # 각 조각의 글자 수
print("조각 길이: 최소", min(lengths), "/ 평균", sum(lengths) // len(lengths), "/ 최대", max(lengths))

이웃한 두 조각이 정말 겹치는지 확인해 봅시다. 조각 1의 끝과 조각 2의 시작을 나란히 출력합니다.

In [ ]:
print("[조각 1의 끝]  ...", chunks[1].page_content[-120:])
print()
print("[조각 2의 시작]", chunks[2].page_content[:120], "...")

조각 1의 마지막 줄이 조각 2의 첫머리에 다시 들어 있습니다. 이것이 `chunk_overlap` 입니다. 덕분에 경계에 걸친 문장도 어느 한 조각에는 온전히 남습니다.

`chunk_size` 를 바꾸면 조각 수가 어떻게 달라지는지도 봅시다.

In [ ]:
# 조각 크기에 따른 조각 수 비교 (overlap은 크기의 10%)
for size in [300, 1000, 2000]:
    n = len(split_documents(docs, chunk_size=size, chunk_overlap=size // 10))
    print(f"chunk_size={size:4d} → 조각 {n}개")

- 조각이 **너무 작으면**: 한 조각에 문맥이 부족합니다(예: "35% increase"만 있고 무엇이 증가했는지 없음).
- 조각이 **너무 크면**: 관련 없는 내용이 많이 섞이고 LLM 입력이 길어집니다.

정답은 없고 데이터에 맞게 실험으로 정합니다. 이 튜토리얼은 1000/100 을 기본으로 씁니다.

## 6. 가장 나이브한 검색: 키워드 겹치기

이제 ① **검색**을 만듭니다. 가장 단순한 방법은 **질문과 조각에 공통으로 들어 있는 단어 수**를 세는 것입니다.
겹치는 단어가 많을수록 관련 있는 조각이라고 보는 것이죠. (XRD 피크 매칭에서 "겹치는 피크 개수"로 후보 상을 고르는 것과 비슷합니다.)

먼저 글을 단어 목록으로 쪼개는 함수를 만듭니다. 이렇게 글을 단어 단위로 자르는 것을 **토큰화(Tokenization)** 라고 합니다.

In [ ]:
import re

# 너무 흔해서 검색에 도움이 안 되는 영어 단어(불용어, stopword)
STOPWORDS = {"the", "of", "in", "and", "a", "an", "to", "is", "was", "for", "with", "by",
             "at", "as", "on", "than", "what", "how", "did", "does", "which", "compared"}

def tokenize(text):
    """글을 소문자 단어 집합으로 바꿉니다. (영어·숫자·한글만 남기고 불용어 제거)"""
    words = re.findall(r"[0-9a-z가-힣]+", text.lower())
    return {w for w in words if w not in STOPWORDS}

print(tokenize("Grain coarsening occurs in the range of 0 to 1.5 at.% Nb"))

`1.5` 가 `1`, `5` 로 쪼개지는 등 완벽하지는 않지만, 단어를 비교하기에는 충분합니다. (집합(set)이라 출력 순서는 매번 다를 수 있습니다.)

In [ ]:
def keyword_score(query, chunk_text):
    """질문과 조각에 공통으로 들어 있는 단어의 개수 = 점수"""
    return len(tokenize(query) & tokenize(chunk_text))  # & 는 교집합

def keyword_search(query, chunks, k=3):
    """모든 조각의 점수를 매겨 높은 순으로 k개를 돌려줍니다."""
    scored = [(keyword_score(query, c.page_content), c) for c in chunks]
    scored.sort(key=lambda pair: pair[0], reverse=True)  # 점수 높은 순 정렬
    return scored[:k]

논문이 영어이므로, 일단 **검색어는 영어로** 적어 봅시다. (한국어 검색어는 7절에서 시험합니다.)

In [ ]:
SEARCH_QUERY = ("In which Nb concentration range does grain coarsening occur in Cu-Nb films, "
                "and how much did the mean grain size increase compared to pure Cu?")
print("검색어의 단어들:", sorted(tokenize(SEARCH_QUERY)))
print()

results = keyword_search(SEARCH_QUERY, chunks, k=3)
for rank, (score, c) in enumerate(results, start=1):
    name = Path(c.metadata["source"]).name[:25]
    print(f"[{rank}위] 점수 {score} | {name}... {c.metadata['page'] + 1}쪽")
    print("     ", c.page_content[:150].replace("\n", " "), "...\n")

In [ ]:
# 정답 문장("0 to 1.5 at.% Nb")이 찾은 조각 안에 들어 있는지 확인
for rank, (score, c) in enumerate(results, start=1):
    print(f"{rank}위 조각에 정답 문장 포함?", "1.5 at.% Nb" in c.page_content)

단 10줄 남짓한 코드로 수많은 조각 중에서 정답 문장이 들어 있는 조각을 찾아냈습니다. 이것이 ① **검색(Retrieval)** 입니다.

## 7. 찾은 조각을 붙여서 다시 질문하기 — 증강과 생성

② **증강(Augmented)**: 찾은 조각들을 "참고자료"로 질문 앞에 붙여 하나의 프롬프트(LLM에게 주는 지시문)를 만듭니다.
"참고자료에 없으면 모른다고 답하라"는 지시도 함께 넣습니다.

In [ ]:
def build_prompt(question, retrieved):
    """검색된 조각들을 참고자료로 붙인 프롬프트를 만듭니다."""
    context = "\n\n".join(f"[발췌 {i}]\n{c.page_content}" for i, (_, c) in enumerate(retrieved, 1))
    return (
        "아래 [논문 발췌]만 근거로 질문에 한국어로 답하세요. "
        "발췌에 없는 내용이면 '자료에서 찾을 수 없습니다'라고 답하세요.\n\n"
        f"[논문 발췌]\n{context}\n\n[질문]\n{question}"
    )

prompt = build_prompt(QUESTION, results)
print("프롬프트 길이:", len(prompt), "글자\n")
print(prompt[:400], "\n...\n", prompt[-250:])

③ **생성(Generation)**: 이 프롬프트를 LLM에게 그대로 보냅니다.

In [ ]:
answer_with_rag = llm.invoke(prompt).content  # 참고자료를 펼쳐 놓고 답하기
print(answer_with_rag)

In [ ]:
# 두 답을 나란히 비교합니다
print("✔ 논문의 정답 : 0 ~ 1.5 at.% Nb 구간에서 결정립 조대화, 10.1 → 13.6 nm (35% 증가)")
print("\n✘ LLM 단독    :", answer_without_rag[:300])
print("\n★ RAG         :", answer_with_rag)

같은 모델, 같은 질문인데 **참고자료를 붙여 주었을 뿐**입니다. 대개 답이 논문의 수치(0 ~ 1.5 at.%, 10.1 → 13.6 nm)에 맞게 바뀝니다. 여러분의 결과에서 위 정답표와 대조해 보세요.
모델을 다시 학습시키지도, 더 큰 모델로 바꾸지도 않았습니다. 이것이 RAG의 핵심입니다.

> 💡 LLM은 이제 "기억해 내기"가 아니라 **"주어진 글 읽고 요약하기"** 를 하고 있습니다. 소형 모델도 읽기·요약은 꽤 잘합니다.
> 또한 답의 근거가 된 조각(파일, 쪽 번호)을 함께 보여 줄 수 있어서, 연구자가 **원문을 직접 확인**할 수 있다는 장점도 있습니다.

## 8. 한계: 한국어로 물으면?

지금까지는 검색어를 영어로 따로 적었습니다. 하지만 우리는 보통 **한국어로 질문**합니다. 같은 뜻의 한국어 질문으로 검색해 봅시다.

In [ ]:
korean_queries = [
    "구리-니오븀 합금에서 결정립이 커지는 농도 구간은?",  # 순수 한국어
    "Cu-Nb 합금에서 결정립이 커지는 농도 구간은?",        # 원소 기호만 영어
]
for q in korean_queries:
    scores = [keyword_score(q, c.page_content) for c in chunks]  # 모든 조각의 점수
    best = max(scores)
    print(f"질문: {q}")
    print(f"  최고 점수 {best} | 최고 점수인 조각 수 {scores.count(best)}개 / 전체 {len(chunks)}개\n")

- **순수 한국어 질문**: 영어 논문과 겹치는 단어가 하나도 없어 **모든 조각이 0점**입니다. 검색이 완전히 실패합니다.
- **원소 기호만 영어**: `cu`, `nb` 두 단어만 겹치므로 Cu와 Nb가 함께 나오는 수많은 조각이 **동점**이 됩니다. 어느 것이 정답 조각인지 구별하지 못합니다.

언어가 같아도 문제는 남습니다. 같은 뜻을 다른 단어로 쓰는 **동의어** 문제입니다.

In [ ]:
# 같은 개념, 다른 표현: 각 단어가 들어 있는 조각 수 세기
for word in ["nb", "niobium", "coarsening", "growth", "resistivity", "resistance"]:
    n = sum(word in tokenize(c.page_content) for c in chunks)
    print(f"{word:12s} → {n:3d}개 조각에 등장")

사람에게 "니오븀 = niobium = Nb", "결정립 조대화 ≈ grain coarsening ≈ grain growth" 는 당연히 같은 뜻이지만,
키워드 겹치기는 **글자가 똑같아야만** 같은 것으로 봅니다. 질문을 영어로 번역하거나 동의어 사전을 만들어 붙일 수도 있지만, 끝이 없습니다.

> 그래서 **글자가 아니라 "의미"로 찾는 방법**이 필요합니다. → 03 노트북의 **임베딩과 벡터 DB**가 바로 이 문제를 풉니다.
> 또 "Nb를 넣으면 무엇이 변하고, 그것이 다시 무엇에 영향을 주는가?" 처럼 **관계를 따라가는 질문**, "Cu보다 비저항이 낮은 금속은?" 처럼 **표에서 정확히 조회할 질문**은 각각 그래프 DB와 관계형 DB가 더 잘합니다.

## 9. 앞으로의 로드맵

이 노트북에서 만든 "키워드 검색 + 프롬프트 붙이기" 가 모든 RAG의 뼈대입니다. 앞으로는 ① 검색 부분을 점점 똑똑하게 바꿔 갑니다.

| # | 노트북 | 무엇을 배우나 | 이 노트북과의 연결 |
|---|---|---|---|
| 00 | 00_Setup | 실습 환경 준비 | — |
| 01 | **01_What_is_RAG** (지금) | LLM의 한계, RAG 뼈대 | 키워드 검색 → 한국어·동의어에 약함 |
| 02 | 02_RDB_SQLite | **표로 찾기**: 관계형 DB, SQL, Text2SQL | 물성표처럼 정확한 수치 조회 |
| 03 | 03_VectorDB | **의미로 찾기**: 임베딩, 벡터 DB | 한국어 질문으로 영어 논문 검색 해결 |
| 04 | 04_GraphDB_Kuzu | **관계로 찾기**: 그래프 DB, Cypher | 소재-공정-물성 관계 따라가기 |
| 05 | 05_Agent | 도구를 쓰는 LLM, 에이전트 | 어떤 DB를 쓸지 LLM이 스스로 결정 |
| 06 | 06_Vector_RAG | **Vector RAG** 구현 | 이 노트북의 검색을 벡터 검색으로 교체 |
| 07 | 07_Graph_RAG | **Graph RAG** 구현 | 검색을 그래프 탐색으로 교체 |
| 08 | 08_Agentic_RAG | **Agentic RAG** 구현 | 여러 검색 방법을 에이전트가 조합 |

## 정리

- **LLM은 다음 단어 예측기**입니다. 흔한 지식은 잘 알지만, 특정 논문의 수치처럼 드문 사실은 그럴듯하게 지어내기도 합니다(**환각**).
- **RAG = 검색(Retrieval) + 증강(Augmented) + 생성(Generation)**. 관련 자료를 찾아 질문에 붙여 주면, 같은 모델로도 근거 있는 답을 얻습니다(오픈북 시험).
- 논문은 페이지 → **조각(Chunk)** 으로 나눠 둡니다. `chunk_size` 는 조각 크기, `chunk_overlap` 은 경계에서 문맥이 끊기지 않게 겹치는 양입니다.
- 가장 단순한 검색은 **키워드 겹치기**입니다. 구현은 쉽지만 **한국어 질문 vs 영어 논문**, **동의어** 문제에 약합니다 → 다음 노트북들에서 해결합니다.

## 직접 해보기

1. `QUESTION` 을 Fe에 대한 질문으로 바꿔 보세요. 예: "Cu-Fe 박막에서 결정립이 커지는 Fe 농도 구간은?" (`SEARCH_QUERY` 도 영어로 함께 바꾸세요.) 논문 정답은 **0 ~ 2.5 at.% Fe** 입니다. LLM 단독 답과 RAG 답을 비교해 보세요.
2. `keyword_search(..., k=1)` 로 조각을 1개만 찾으면 답이 어떻게 달라지나요? `k=6` 처럼 많이 주면요? (힌트: 정답 조각이 몇 위였는지 떠올려 보세요.)
3. 리뷰 논문(Sung et al., 2026)에 대해 물어보세요. 예: 검색어 `"electron mean free path of Cu in nm"` 으로 검색한 뒤 "Cu의 전자 평균자유행로(EMFP)는 몇 nm인가요?" 라고 질문해 보세요. (논문의 값: 약 **39 nm**)